In [ ]:
# ============================================================
# EXPERIMENT 5A : DECISION TREE CLASSIFICATION FROM SCRATCH
# ============================================================
import numpy as np
from collections import Counter

class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTreeClassifierScratch:
    def __init__(self,
        max_depth=3,
        min_samples_split=2
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    # -----------------------------------------
    # Calculate Entropy
    # -----------------------------------------
    def entropy(self, y):
        if len(y) == 0:
            return 0

        counts = np.bincount(y)
        probabilities = (
            counts[counts > 0] / len(y)
        )
        entropy = -np.sum(
            probabilities *
            np.log2(probabilities)
        )
        return entropy

    # -----------------------------------------
    # Calculate Information Gain
    # -----------------------------------------
    def information_gain(
        self,
        y,
        y_left,
        y_right
    ):
        parent_entropy = self.entropy(y)
        n = len(y)
        n_left = len(y_left)
        n_right = len(y_right)

        if (
            n_left == 0 or
            n_right == 0
        ):
            return 0

        weighted_entropy = (
            (n_left / n) *
            self.entropy(y_left)
            +
            (n_right / n) *
            self.entropy(y_right)
        )

        gain = (
            parent_entropy -
            weighted_entropy
        )
        return gain

    # --------------------------------------------
    # Find Best Split
    # --------------------------------------------
    def best_split(self, X, y):
        best_gain = -1
        best_feature = None
        best_threshold = None
        n_samples, n_features = X.shape

        for feature in range(n_features):
            feature_values = X[:, feature]
            thresholds = np.unique(
                feature_values
            )

            for threshold in thresholds:
                left_indices = (
                    feature_values <= threshold
                )
                right_indices = (
                    feature_values > threshold
                )

                y_left = y[left_indices]
                y_right = y[right_indices]

                if (
                    len(y_left) == 0 or
                    len(y_right) == 0
                ):
                    continue

                gain = self.information_gain(
                    y,
                    y_left,
                    y_right
                )

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return (
            best_feature,
            best_threshold,
            best_gain
        )

    # Build Decision Tree
    def build_tree(
        self,
        X,
        y,
        depth=0
    ):
        n_samples = len(y)
        n_classes = len(
            np.unique(y)
        )

        if (
            depth >= self.max_depth
            or n_classes == 1
            or n_samples < self.min_samples_split
        ):
            leaf_value = Counter(
                y
            ).most_common(1)[0][0]
            return Node(
                value=leaf_value
            )

        (
            feature,
            threshold,
            gain
        ) = self.best_split(
            X,
            y
        )

        if (
            feature is None
            or gain <= 0
        ):
            # NOTE: PDF shows most_common(1)[0][1] here,
            # which appears to be an extraction/typo error.
            # The correct majority-class label is [0][0].
            leaf_value = Counter(
                y
            ).most_common(1)[0][0]
            return Node(
                value=leaf_value
            )

        left_indices = X[:, feature] <= threshold
        right_indices = X[:, feature] > threshold

        left = self.build_tree(
            X[left_indices],
            y[left_indices],
            depth + 1
        )
        right = self.build_tree(
            X[right_indices],
            y[right_indices],
            depth + 1
        )

        return Node(
            feature=feature,
            threshold=threshold,
            left=left,
            right=right
        )

    # Fit
    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=int)
        self.root = self.build_tree(
            X,
            y
        )

    # --------------------------------------------
    # Predict One Sample
    # --------------------------------------------
    def predict_one(self, x, node):
        if node.value is not None:
            return node.value

        if (
            x[node.feature]
            <= node.threshold
        ):
            return self.predict_one(
                x,
                node.left
            )

        return self.predict_one(
            x,
            node.right
        )

    # --------------------------------------------
    # Predict Multiple Samples
    # --------------------------------------------
    def predict(self, X):
        X = np.array(X, dtype=float)
        predictions = []

        for x in X:
            predictions.append(
                self.predict_one(
                    x,
                    self.root
                )
            )

        return np.array(predictions)

# ===============================================================
# Dataset
# ===============================================================
X = np.array([
    [1, 55],
    [2, 60],
    [3, 65],
    [4, 70],
    [5, 75],
    [6, 80],
    [7, 85],
    [8, 90]
])

y = np.array([
    0, 0, 0, 1,
    1, 1, 1, 1
])

# ===============================================================
# Train-Test Split
# ===============================================================
X_train = X[:6]
y_train = y[:6]

X_test = X[6:]
y_test = y[6:]

print("Training Data:")
print(X_train)

print("\nTraining Labels:")
print(y_train)

print("\nTesting Data:")
print(X_test)

print("\nTesting Labels:")
print(y_test)

# ============================================================
# Create Model
# ============================================================
tree = DecisionTreeClassifierScratch(
    max_depth=3
)

# ============================================================
# Initial Entropy
# ============================================================
initial_entropy = tree.entropy(
    y_train
)

print("\nInitial Entropy:")
print(
    round(
        initial_entropy,
        4
    )
)

# ============================================================
# Best Split
# ============================================================
(
    feature,
    threshold,
    gain
) = tree.best_split(
    X_train,
    y_train
)

print("\nBest Split")
print("------------------")
print(
    "Feature Index :",
    feature
)
print(
    "Threshold :",
    threshold
)
print(
    "Information Gain :",
    round(
        gain,
        4
    )
)

# ============================================================
# Train
# ============================================================
tree.fit(
    X_train,
    y_train
)

# ============================================================
# Prediction
# ============================================================
y_pred = tree.predict(
    X_test
)

# ============================================================
# Results
# ============================================================
print("\nActual Labels :")
print(y_test)

print("\nPredicted Labels :")
print(y_pred)

# ============================================================
# Accuracy
# ============================================================
accuracy = np.mean(
    y_test == y_pred
)

print(
    "\nAccuracy :",
    round(
        accuracy,
        4
    )
)
print(
    "Accuracy (%):",
    round(
        accuracy * 100,
        2
    )
)

In [ ]:
# ============================================================
# EXPERIMENT 5B : DECISION TREE REGRESSION FROM SCRATCH
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Step 1: Create Synthetic Dataset
# ============================================================
# X = Study Hours, y = Marks
X = np.array([
    [1.0], [1.5], [2.0], [2.5], [3.0],
    [3.5], [4.0], [4.5], [5.0], [5.5],
    [6.0], [6.5], [7.0], [7.5], [8.0],
    [8.5], [9.0], [9.5], [10.0], [10.5]
], dtype=float)

y = np.array([
    35, 38, 40, 45, 50,
    52, 55, 60, 65, 68,
    70, 72, 75, 78, 85,
    86, 88, 90, 95, 96
], dtype=float)

print("Dataset Shape:", X.shape)

# ============================================================
# Step 2: Train-Test Split From Scratch
# ============================================================
np.random.seed(42)

indices = np.random.permutation(len(X))
train_size = int(0.80 * len(X))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train, y_train = X[train_indices], y[train_indices]
X_test, y_test = X[test_indices], y[test_indices]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# ============================================================
# Step 3: Tree Node
# ============================================================
class Node:
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature = feature        # splitting feature index
        self.threshold = threshold    # splitting threshold
        self.left = left              # left subtree
        self.right = right            # right subtree
        self.value = value            # leaf prediction (mean)

# ============================================================
# Step 4: Decision Tree Regressor From Scratch
# ============================================================
class DecisionTreeRegressorScratch:

    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    # ----------------------------
    # Mean Squared Error
    # ----------------------------
    def mse(self, y):
        if len(y) == 0:
            return 0
        mean_value = np.mean(y)
        return np.mean((y - mean_value) ** 2)

    # ----------------------------
    # Weighted MSE
    # ----------------------------
    def weighted_mse(self, y_left, y_right):
        n = len(y_left) + len(y_right)
        if n == 0:
            return 0
        return (
            (len(y_left) / n) * self.mse(y_left)
            + (len(y_right) / n) * self.mse(y_right)
        )

    # ----------------------------
    # Find Best Split
    # ----------------------------
    def best_split(self, X, y):
        best_mse = np.inf
        best_feature = None
        best_threshold = None

        n_samples, n_features = X.shape

        for feature in range(n_features):
            feature_values = X[:, feature]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                left_indices = feature_values <= threshold
                right_indices = feature_values > threshold

                y_left = y[left_indices]
                y_right = y[right_indices]

                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                split_mse = self.weighted_mse(y_left, y_right)

                if split_mse < best_mse:
                    best_mse = split_mse
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold, best_mse

    # ----------------------------
    # Build Tree (Recursion)
    # ----------------------------
    def build_tree(self, X, y, depth=0):
        n_samples = len(y)

        if n_samples == 0:
            return Node(value=0.0)

        # Stopping conditions -> leaf node
        if (depth >= self.max_depth
                or n_samples < self.min_samples_split
                or np.all(y == y[0])):
            return Node(value=np.mean(y))

        feature, threshold, split_mse = self.best_split(X, y)

        # No valid split -> leaf node
        if feature is None:
            return Node(value=np.mean(y))

        left_indices = X[:, feature] <= threshold
        right_indices = X[:, feature] > threshold

        left_child = self.build_tree(X[left_indices], y[left_indices], depth + 1)
        right_child = self.build_tree(X[right_indices], y[right_indices], depth + 1)

        return Node(feature=feature, threshold=threshold,
                    left=left_child, right=right_child)

    # ----------------------------
    # Train
    # ----------------------------
    def fit(self, X, y):
        self.root = self.build_tree(X, y, depth=0)

    # ----------------------------
    # Predict One Sample
    # ----------------------------
    def predict_one(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self.predict_one(x, node.left)
        else:
            return self.predict_one(x, node.right)

    # ----------------------------
    # Predict All Samples
    # ----------------------------
    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in X])

# ============================================================
# Step 5: Create and Train the Model
# ============================================================
model = DecisionTreeRegressorScratch(max_depth=3, min_samples_split=2)
model.fit(X_train, y_train)

print("\nDecision Tree Regression Model Trained Successfully.")

# ============================================================
# Step 6: Display Tree Structure
# ============================================================
def print_tree(node, indent=""):
    if node.value is not None:
        print(indent + "Leaf -> Prediction:", round(node.value, 4))
        return
    print(indent + f"X[{node.feature}] <= {node.threshold}")
    print(indent + "Left:")
    print_tree(node.left, indent + "  ")
    print(indent + "Right:")
    print_tree(node.right, indent + "  ")

print("\n===================================")
print("TREE STRUCTURE")
print("===================================")
print_tree(model.root)

# ============================================================
# Step 7: Make Predictions
# ============================================================
y_pred = model.predict(X_test)

print("\n===================================")
print("ACTUAL VS PREDICTED")
print("===================================")
print("Hours\tActual\tPredicted")
for i in range(len(y_test)):
    print(f"{X_test[i, 0]:.1f}\t{y_test[i]:.2f}\t{y_pred[i]:.2f}")

# ============================================================
# Step 8: Performance Measures
# ============================================================
errors = y_test - y_pred

mae = np.mean(np.abs(errors))
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)

ss_res = np.sum(errors ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("\n===================================")
print("DECISION TREE REGRESSION PERFORMANCE")
print("===================================")
print("MAE :", round(mae, 4))
print("MSE :", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("R2  :", round(r2, 4))

# ============================================================
# Step 9: Predict a New Observation
# ============================================================
new_hours = 6.2
new_prediction = model.predict(np.array([[new_hours]]))[0]

print("\n===================================")
print("NEW PREDICTION")
print("===================================")
print("Study Hours:", new_hours)
print("Predicted Marks:", round(new_prediction, 2))

# ============================================================
# Step 10: Plot Actual vs Tree Prediction
# ============================================================
X_grid = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_grid = model.predict(X_grid)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, label="Actual Data")
plt.plot(X_grid, y_grid, label="Decision Tree Prediction")
plt.scatter(new_hours, new_prediction, marker="x", s=120,
            label="New Prediction")
plt.xlabel("Study Hours")
plt.ylabel("Marks")
plt.title("Decision Tree Regression From Scratch")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()